## Appendix B: How to Run Code

This notebook walks through every stage of the project:
1. B.1 Requirements 
2. B.2 Google Colab Setup (skip if running locally)
3. B.3 Dataset
4. B.4 Training a Single MOdel 
5. B.5 Evaluating a Model 
6. B.6 Visualisation 
7. B.7 Comparing all architectures

### B.1 Requirements

In [ ]:
# Install dependencies (only needed once):
#   pip install torch torchvision matplotlib kagglehub ipython
#
# On Google Colab, also run:
#   !pip install kagglehub

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import os

from utils import (
    import_dataset,
    initialize_dataset,
    split_dataset,
    initialize_dataloaders,
    train_model,
    evaluate_model,
    save_model,
    visualize_dataset,
    plot_train_val,
    plot_anomalies,
    plot_anomaly_distribution,
    multiplot_train_val,
)
from compare import compare_architecture

# Reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# B.1 — Verify CUDA
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())  # True if CUDA is ready

use_cuda = True
device = torch.device("cuda" if (use_cuda and torch.cuda.is_available()) else "cpu")
print("Using device:", device)

### B.2 Google Colab Setup

**Skip this section if you are running locally.**

Steps:
1. Upload example.ipynb, compare.py, models/, and utils/ to a single folder in Google Drive.
2. Open example.ipynb via **File → Open notebook → Google Drive**.
3. Set the runtime via **Runtime → Change runtime type → T4 GPU**.
4. Run the cell below to mount Drive and navigate to the project folder.

In [ ]:
# Google Colab only: mount Drive and change to project directory
# Comment out or skip this cell when running locally.
# run this cell before importing utils at the start
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    # Update this path to match where you uploaded the project in Drive
    PROJECT_PATH = "/content/drive/MyDrive/DiabeticRetinopathyDL"
    os.chdir(PROJECT_PATH)
    print("Working directory:", os.getcwd())

    os.system("pip install kagglehub")
else:
    print("Not running on Colab — skipping Drive mount.")


<a id='B3'></a>
### B.3 Dataset

import_dataset() downloads the Kaggle dataset on the **first run only** and copies it to ./dataset/. Subsequent runs detect the existing folder and skip the download.

The dataset contains two classes:
- **Healthy** — 1 000 retinal images
- **Severe DR** — 190 retinal images

The 80/10/10 stratified split ensures class balance across Train / Test / Validation.

In [ ]:
#Download dataset (nothing happens if ./dataset already exists)
import_dataset()

In [ ]:
#Load and split dataset
full_dataset = initialize_dataset()
train_dataset, test_dataset, validation_dataset = split_dataset(full_dataset)

In [ ]:
#Create DataLoaders (batch_size=32 by default)
torch.manual_seed(42)
train_dataloader, test_dataloader, validation_dataloader = initialize_dataloaders(
    train_dataset, test_dataset, validation_dataset
)

print(f"Train batches : {len(train_dataloader)}")
print(f"Test batches  : {len(test_dataloader)}")
print(f"Val batches   : {len(validation_dataloader)}")

### B.4 Training a Single Model

All six architectures share the same training interface — train_model().  
Swap the class to train a different architecture:

| Architecture | Class |
|---|---|
| U-Net (baseline) | UNetClassifier |
| Attention U-Net | AUNetClassifier |
| Residual U-Net | ResUNetClassifier |
| Attention + Residual U-Net | AResUNetClassifier |
| EfficientNet-B0 (pretrained) | EfficientNetB0Classifier |
| U-Net without skip connections | NoSkipUNetClassifier |


When verbose=True (default), it prints per-epoch metrics and plots loss / accuracy curves at the end.

In [ ]:
# Example 1: Train the baseline U-Net
from models import UNetClassifier

model = UNetClassifier().to(device)
criterion = nn.CrossEntropyLoss()

train_loss, train_acc, val_loss, val_acc = train_model(
    model,
    criterion,
    train_dataloader,
    validation_dataloader,
    device,
    lr=0.001,
    num_epochs=20,
    verbose=True,
)

In [ ]:
# Example 2: Train the Attention U-Net
from models import AUNetClassifier

model_aunet = AUNetClassifier().to(device)
criterion = nn.CrossEntropyLoss()

train_loss_aunet, train_acc_aunet, val_loss_aunet, val_acc_aunet = train_model(
    model_aunet,
    criterion,
    train_dataloader,
    validation_dataloader,
    device,
)

In [ ]:
# Example 3: Train the Residual U-Net
from models import ResUNetClassifier

model_resunet = ResUNetClassifier().to(device)
criterion = nn.CrossEntropyLoss()

train_loss_resunet, train_acc_resunet, val_loss_resunet, val_acc_resunet = train_model(
    model_resunet,
    criterion,
    train_dataloader,
    validation_dataloader,
    device,
)

In [ ]:
# Example 4: Train the Attention + Residual U-Net
from models import AResUNetClassifier

model_aresunet = AResUNetClassifier().to(device)
criterion = nn.CrossEntropyLoss()

train_loss_aresunet, train_acc_aresunet, val_loss_aresunet, val_acc_aresunet = train_model(
    model_aresunet,
    criterion,
    train_dataloader,
    validation_dataloader,
    device,
)

In [ ]:
# Example 5: Train EfficientNet-B0 (pretrained backbone)
from models import EfficientNetB0Classifier

model_effnet = EfficientNetB0Classifier().to(device)
criterion = nn.CrossEntropyLoss()

train_loss_effnet, train_acc_effnet, val_loss_effnet, val_acc_effnet = train_model(
    model_effnet,
    criterion,
    train_dataloader,
    validation_dataloader,
    device,
)

In [ ]:
# Example 6: Train U-Net without skip connections
from models import NoSkipUNetClassifier

model_noskip = NoSkipUNetClassifier().to(device)
criterion = nn.CrossEntropyLoss()

train_loss_noskip, train_acc_noskip, val_loss_noskip, val_acc_noskip = train_model(
    model_noskip,
    criterion,
    train_dataloader,
    validation_dataloader,
    device,
)

### Saving a trained model

save_model() persists a model to disk:

Files are named "{name}\_{epoch}\_{kwargs}-{date}".pt, e.g. "UNetClassifier_20_lr0.001-20260418.pt".

In [ ]:
# save the baseline U-Net trained above
# change the model name accordingly if you want to save a different model
save_model(model, "UNetClassifier", epoch=20, dir="./outputs/models", lr=0.001)

### Loading a saved model

In [ ]:
# load a previously saved model
model_path = "./outputs/models/UNetClassifier_20_lr0.001-20260418.pt"
loaded_model = torch.load(model_path, weights_only=False)
loaded_model = loaded_model.to(device)
print("Loaded:", loaded_model._name)

### B.5 Evaluating a Model

evaluate_model() returns four metrics:
- **Accuracy** — overall classification accuracy (%)
- **F1 Score** — harmonic mean of precision and recall
- **F2 Score** — weighted toward recall (important for medical screening)
- **Anomaly Detection** — recall for the Severe DR class (%)

U-Net-based models also save their decoder output images to output_dir.

In [ ]:
#evaluate the baseline U-Net on all three splits
train_acc_score, train_f1, train_f2, train_recall = evaluate_model(
    model, train_dataloader, output_dir="./outputs/UNet/train", device=device
)
val_acc_score, val_f1, val_f2, val_recall = evaluate_model(
    model, validation_dataloader, output_dir="./outputs/UNet/val", device=device
)
test_acc_score, test_f1, test_f2, test_recall = evaluate_model(
    model, test_dataloader, output_dir="./outputs/UNet/test", device=device
)

print(f"Train | Accuracy: {train_acc_score:.2f}% | F1: {train_f1:.2f} | F2: {train_f2:.2f} | Anomaly Detection: {train_recall:.2f}%")
print(f"  Val | Accuracy: {val_acc_score:.2f}% | F1: {val_f1:.2f} | F2: {val_f2:.2f} | Anomaly Detection: {val_recall:.2f}%")
print(f" Test | Accuracy: {test_acc_score:.2f}% | F1: {test_f1:.2f} | F2: {test_f2:.2f} | Anomaly Detection: {test_recall:.2f}%")

### B.6 Visualisation

In [ ]:
# display sample images from each class
visualize_dataset()

In [ ]:
# plot training and validation loss / accuracy curves
# uses the lists returned by train_model() above
plot_train_val(train_loss, train_acc, val_loss, val_acc)

In [ ]:
#plot anomaly detection outputs side-by-side with original inputs
# requires evaluate_model() to have run first so that output images exist
plot_anomalies(
    validation_dataset,
    test_dataset,
    val_dir="./outputs/UNet/val",
    test_dir="./outputs/UNet/test",
)

In [ ]:
# overlay training curves from multiple models on one plot
# build metrics dict by training or loading multiple models first
metrics = {
    "UNet": {
        "train": {"loss": train_loss, "acc": train_acc},
        "val":   {"loss": val_loss,   "acc": val_acc},
    },
    # add further models here following the same structure:
    # "AUNet": {"train": {"loss": train_loss_aunet, "acc": train_acc_aunet},
    #           "val":   {"loss": val_loss_aunet,   "acc": val_acc_aunet}},
    # "ResUNet": {"train": {"loss": train_loss_resunet, "acc": train_acc_resunet},
    #             "val":   {"loss": val_loss_resunet,   "acc": val_acc_resunet}},
}

multiplot_train_val(metrics, save=False)


### B.7 Comparing All Architectures

compare_architecture() trains (or reloads) all six models and prints a unified metric table.  
Set reload=True to force re-training even if saved weights already exist.

In [ ]:
# Benchmark all six architectures
# reload=False  → reuse saved models if available (faster)
# reload=True   → always retrain from scratch
compare_architecture(
    train_dataloader,
    validation_dataloader,
    test_dataloader,
    device,
    reload=False,
)